In [ ]:
import pandas as pd
import numpy as np
import re

# 1. Carga inicial del dataset
df = pd.read_csv('boxeadores.csv', encoding='latin1')

print(f"Dataset original: {df.shape[0]} filas, {df.shape[1]} columnas.")

# ==============================================================================
# PASO 1: Limpieza y tipado de columnas numéricas y monetarias
# ==============================================================================

# Limpiar 'ganancias_usd': eliminar símbolos de moneda y comas, convertir a float
df['ganancias_usd'] = (
    df['ganancias_usd']
    .astype(str)
    .str.replace(r'[\$,]', '', regex=True)  # Elimina '$' y ','
    .astype(float)
)
# Las ganancias negativas no tienen sentido físico, las convertimos a NaN
df['ganancias_usd'] = np.where(df['ganancias_usd'] < 0, np.nan, df['ganancias_usd'])

# Corregir errores de digitación obvios en estatura (ej. 18.8 cm -> 188 cm)
df['estatura_cm'] = np.where(df['estatura_cm'] < 100, df['estatura_cm'] * 10, df['estatura_cm'])

# Las victorias, derrotas, empates y peso no pueden ser negativos. Si lo son, son NaN.
cols_estadisticas = ['victorias', 'derrotas', 'empates', 'peso_kg']
for col in cols_estadisticas:
    df[col] = np.where(df[col] < 0, np.nan, df[col])

# ==============================================================================
# PASO 2: Estandarización de variables categóricas
# ==============================================================================

# Mapeo de países (de códigos ISO o nombres sueltos a nombres completos en español)
map_paises = {
    'Mex': 'México', 'UK': 'Reino Unido', 'JPN': 'Japón',
    'USA': 'Estados Unidos', 'RUS': 'Rusia'
}
df['pais_origen'] = df['pais_origen'].replace(map_paises)

# Mapeo y estandarización de la guardia
df['guardia'] = df['guardia'].astype(str).str.strip().str.lower()
map_guardia = {
    'ortodoxo': 'Ortodoxo', 'zurdo': 'Zurdo', 'ambi': 'Ambidiestro', 'nan': 'Desconocido'
}
df['guardia'] = df['guardia'].replace(map_guardia)

# Estandarización de categorías de peso (Title Case y corrección de sinónimos)
df['categoria_peso'] = df['categoria_peso'].astype(str).str.strip().str.title()
map_categorias = {
    'Super Welter': 'Super Welter', 'Mediano': 'Mediano', 'Pluma': 'Pluma',
    'Crucero': 'Crucero', 'Super Pluma': 'Super Pluma', 'Super Gallo': 'Super Gallo',
    'Moscas': 'Mosca', 'Gallo': 'Gallo', 'Ligero': 'Ligero', 'Semi Pesado': 'Semi Pesado',
    'Pesado': 'Pesado', 'Super Mediano': 'Super Mediano', 'Unknown': 'Desconocido'
}
df['categoria_peso'] = df['categoria_peso'].replace(map_categorias)

# ==============================================================================
# PASO 3: Manejo de Fechas
# ==============================================================================

# Convertir a datetime manejando formatos mixtos y dejando NaT (Not a Time) si falla
df['fecha_debut'] = pd.to_datetime(df['fecha_debut'], format='mixed', dayfirst=True, errors='coerce')

# ==============================================================================
# PASO 4: Imputación de Valores Faltantes (Missing Values)
# ==============================================================================

# En boxeo, si no se registran derrotas o empates, es seguro asumir que son 0 (especialmente en activos)
df['derrotas'] = df['derrotas'].fillna(0).astype(int)
df['empates'] = df['empates'].fillna(0).astype(int)

# Imputar peso y estatura con la mediana de su propia categoría de peso (más inteligente que la mediana global)
df['peso_kg'] = df.groupby('categoria_peso')['peso_kg'].transform(lambda x: x.fillna(x.median()))
df['estatura_cm'] = df.groupby('categoria_peso')['estatura_cm'].transform(lambda x: x.fillna(x.median()))

# Si aún queda algún NaN en peso/estatura (por categorías vacías), usamos la mediana global
df['peso_kg'] = df['peso_kg'].fillna(df['peso_kg'].median())
df['estatura_cm'] = df['estatura_cm'].fillna(df['estatura_cm'].median())

# Imputar ganancias con la mediana general
df['ganancias_usd'] = df['ganancias_usd'].fillna(df['ganancias_usd'].median())

# ==============================================================================
# PASO 5: Resolución de Entidades (Deduplicación Inteligente)
# ==============================================================================

# Crear una versión normalizada del nombre para comparar (minúsculas, sin espacios extra)
df['nombre_norm'] = df['nombre'].astype(str).str.lower().str.strip()

# Eliminar filas donde el nombre es completamente desconocido (no se puede resolver)
df = df.dropna(subset=['nombre'])
df = df[df['nombre'].str.strip() != '']

# Ordenar por victorias de mayor a menor para conservar el registro más completo/exitoso en caso de duplicado
df = df.sort_values(by='victorias', ascending=False)

# Eliminar duplicados basados en el nombre normalizado, manteniendo el primero (el de más victorias)
df = df.drop_duplicates(subset=['nombre_norm'], keep='first')

# Limpiar columnas auxiliares y resetear índice
df = df.drop(columns=['nombre_norm']).reset_index(drop=True)
df['id_boxeador'] = range(1, len(df) + 1)  # Reasignar IDs secuenciales

# ==============================================================================
# PASO 6: Guardado y Verificación
# ==============================================================================

df.to_csv('boxeadores_limpio.csv', index=False, encoding='latin1')

print(f"Dataset limpio: {df.shape[0]} filas, {df.shape[1]} columnas.")
print("\nMuestra de los datos limpios:")
print(df[['nombre', 'categoria_peso', 'pais_origen', 'peso_kg', 'estatura_cm', 'guardia']].head(10))

📊 Dataset original: 73 filas, 12 columnas.
✅ Dataset limpio: 61 filas, 12 columnas.

🔍 Muestra de los datos limpios:
                  nombre categoria_peso     pais_origen  peso_kg  estatura_cm  \
0     JULIO CÉSAR CHÁVEZ   Super Ligero          México    635.0        171.0   
1         Manny Pacquiao         Welter       Filipinas     66.7        166.0   
2  Saúl 'Canelo' Álvarez  Super Mediano          México     76.2        173.0   
3             Mike Tyson         Pesado  Estados Unidos    100.0        178.0   
4   Floyd Mayweather Jr.         Welter  Estados Unidos     66.7        173.0   
5      juan 'ful' fulton          Mosca        Colombia     50.0        169.0   
6       Juan 'Ber' Berto    Semi Pesado           Japón    107.6        189.0   
7    Vergil 'Rom' Romero  Super Mediano  Estados Unidos     67.1        183.0   
8      Luis 'Val' Valdez         Ligero           Japón     60.9        178.0   
9         Deontay Wilder         Pesado  Estados Unidos     98.0        2